# Практическая работа №3. Распределённая модель обработки данных

**Часть 1. Учебная последовательность на данных ecommerce.sqlite**

Цель — освоить базовый цикл распределённой обработки табличных данных средствами Apache Spark и PySpark: подготовка источника, чтение в DataFrame, работа со схемой и типами, обработка пропусков, агрегации, соединения, Spark SQL и запись результата.

## Версии окружения

In [1]:
import os
import sys
import subprocess

import pyspark

java = subprocess.run(["java", "-version"], capture_output=True, text=True)

print("Python:", sys.version.split()[0])
print("Java:", java.stderr.splitlines()[0])
print("PySpark:", pyspark.__version__)

Python: 3.11.16
Java: java version "21.0.9" 2025-10-21 LTS
PySpark: 4.2.0


## 3.2. Извлечение таблиц SQLite в staging-каталог

In [2]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("ecommerce.sqlite")
STAGING = Path("spark_input")
STAGING.mkdir(exist_ok=True)

TABLES = [
    "orders",
    "order_items",
    "products",
    "customers",
    "order_payments",
    "order_reviews",
    "product_category_name_translation",
]

with sqlite3.connect(DB_PATH) as conn:
    for table in TABLES:
        part = pd.read_sql_query(
            f'SELECT * FROM "{table}"',
            conn
        )
        part.to_csv(
            STAGING / f"{table}.csv",
            index=False,
            encoding="utf-8"
        )
        print(f"{table:36s} {len(part):>8} rows")

orders                                  99441 rows
order_items                            112650 rows
products                                32951 rows
customers                               99441 rows
order_payments                         103886 rows
order_reviews                           97621 rows
product_category_name_translation          71 rows


**Вывод.** Семь таблиц базы выгружены в CSV-файлы каталога `spark_input`, который Spark читает без внешнего JDBC-драйвера. Наибольшая из выгруженных таблиц — `order_items` (112 650 строк); таблица `geolocation` не извлекалась, так как в работе не используется.

## 4.1. Создание SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("Practice3_Ecommerce")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)
print("Java (Spark):", spark.sparkContext._jvm.System.getProperty("java.version"))

Master: local[*]
App: Practice3_Ecommerce
Spark: 4.2.0
Java (Spark): 21.0.9


**Вывод.** Создана локальная сессия Spark в режиме `local[*]`: вычисления распределяются по логическим ядрам одного компьютера, при этом сохраняется программная модель Spark (разделы, план вычислений, задачи). Число разделов после перераспределяющих операций уменьшено до 8, что соответствует небольшому объёму учебных данных.

## 5.1. Чтение orders без автоматического вывода типов

In [4]:
orders_raw = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(str(STAGING / "orders.csv"))
)

orders_raw.printSchema()
print("Строк:", orders_raw.count())

orders_raw.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
).show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

Строк: 99441
+--------------------------------+--------------------------------+------------+------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|
+--------------------------------+--------------------------------+------------+------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |
|47770eb9100c2d0c44946d9cf07ec65d|41c

**Вывод.** Таблица `orders` содержит 99 441 строку; без указания схемы все восемь столбцов прочитаны как `string`. Для идентификаторов это допустимо, однако временные отметки перед вычислениями требуется привести к типу `timestamp`.

## 5.2. Явная схема для order_items

In [5]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

items = (
    spark.read
    .option("header", True)
    .schema(order_items_schema)
    .csv(str(STAGING / "order_items.csv"))
)

items.printSchema()
print("Строк:", items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Строк: 112650


**Вывод.** Явная схема `StructType` сразу задала числовые типы для `order_item_id`, `price` и `freight_value` (112 650 строк), поэтому арифметические операции выполняются без дополнительного приведения. В отличие от `inferSchema`, явная схема не требует отдельного прохода по данным и фиксирует контракт входной таблицы.

## 5.3. Преобразование временных полей orders

In [6]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = orders_raw
for c in timestamp_cols:
    orders = orders.withColumn(
        c,
        F.to_timestamp(F.col(c), "yyyy-MM-dd HH:mm:ss")
    )

orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



**Вывод.** Пять временных полей преобразованы в `timestamp`. Исходный `orders_raw` не изменился: `withColumn()` создаёт новый DataFrame, что отражает неизменяемость объектов Spark.

## 6.1. Фильтрация и вычисляемый признак стоимости позиции

In [7]:
expensive_items = (
    items
    .filter(F.col("price") >= 500)
    .withColumn(
        "gross_item_value",
        F.round(F.col("price") + F.col("freight_value"), 2)
    )
    .select(
        "order_id", "order_item_id",
        "price", "freight_value", "gross_item_value"
    )
)

print("Позиций с price >= 500:", expensive_items.count())

expensive_items.orderBy(
    F.desc("gross_item_value")
).show(5, truncate=False)

Позиций с price >= 500: 3239
+--------------------------------+-------------+------+-------------+----------------+
|order_id                        |order_item_id|price |freight_value|gross_item_value|
+--------------------------------+-------------+------+-------------+----------------+
|0812eb902a67711a1cb742b3cdaa65ae|1            |6735.0|194.31       |6929.31         |
|fefacc66af859508bf1a7934eab1e97f|1            |6729.0|193.21       |6922.21         |
|f5136e38d1a14a4dbd87dff67da82701|1            |6499.0|227.66       |6726.66         |
|a96610ab360d42a2e5335a3998b4718a|1            |4799.0|151.34       |4950.34         |
|199af31afc78c699f0dbf71fb178d4d4|1            |4690.0|74.34        |4764.34         |
+--------------------------------+-------------+------+-------------+----------------+
only showing top 5 rows


**Вывод.** Условию `price ≥ 500` соответствуют 3 239 из 112 650 позиций; максимальная стоимость позиции с доставкой составляет 6 929,31. Признак `gross_item_value` относится к отдельной позиции, а не ко всему заказу, так как один заказ может содержать несколько позиций.

## 6.2. Временные преобразования заказов

In [8]:
delivered = (
    orders
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "is_late",
        F.col("order_delivered_customer_date") >
        F.col("order_estimated_delivery_date")
    )
)

with_delivery_date = delivered.filter(
    F.col("order_delivered_customer_date").isNotNull()
)
late = with_delivery_date.filter(F.col("is_late"))

print("Доставленных заказов:", delivered.count())
print("С известной датой доставки:", with_delivery_date.count())
print("Позже ожидаемой даты:", late.count())

Доставленных заказов: 96478
С известной датой доставки: 96470
Позже ожидаемой даты: 7826


**Вывод.** Из 96 478 заказов со статусом `delivered` фактическая дата доставки известна для 96 470. В 7 826 случаях (около 8,1%) заказ доставлен позже ожидаемой даты.

## 7.1. Подсчёт пропусков во временных полях orders

In [9]:
null_counts = orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in timestamp_cols
])

null_counts.show(vertical=True)

-RECORD 0-----------------------------
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



**Вывод.** Дата покупки и ожидаемая дата доставки заполнены полностью. Больше всего пропусков у фактической даты доставки клиенту (2 965), поэтому перед любой обработкой проверено, с чем они связаны.

## 7.2. Предметная проверка пропусков по статусам

In [10]:
missing_delivery_by_status = (
    orders
    .filter(F.col("order_delivered_customer_date").isNull())
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

missing_delivery_by_status.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |619  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|delivered   |8    |
|created     |5    |
|approved    |2    |
+------------+-----+



**Вывод.** Из 2 965 заказов без даты доставки только 8 имеют статус `delivered`; остальные относятся к отправленным, отменённым, недоступным и промежуточным статусам. Следовательно, пропуск отражает незавершённый процесс, и удаление таких строк исказило бы данные о заказах.

## 7.3. Заполнение пропуска категориального признака

In [11]:
products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "products.csv"))
)

missing_categories = products.filter(
    F.col("product_category_name").isNull()
).count()

products_clean = products.fillna({
    "product_category_name": "unknown"
})

print("Товаров без категории:", missing_categories)

Товаров без категории: 610


**Вывод.** У 610 из 32 951 товара отсутствует категория. Пропуск заполнен значением `unknown`: это не создаёт фиктивной числовой величины, а выделяет такие товары в отдельную группу, которая сохранится при соединении с позициями заказов.

## 8.1. Распределение заказов по статусам

In [12]:
status_stats = (
    orders
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

status_stats.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



**Вывод.** Статус `delivered` имеют 96 478 из 99 441 заказа (97%), остальные семь статусов встречаются редко. Поэтому общая статистика по доставке в основном описывает завершённые заказы.

## 8.2. Несколько агрегатных функций для способов оплаты

In [13]:
payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "order_payments.csv"))
)

payment_stats = (
    payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("payments"),
        F.round(F.sum("payment_value"), 2).alias("total_value"),
        F.round(F.avg("payment_value"), 2).alias("avg_value")
    )
    .orderBy(F.desc("total_value"))
)

payment_stats.show(truncate=False)

+------------+--------+-------------+---------+
|payment_type|payments|total_value  |avg_value|
+------------+--------+-------------+---------+
|credit_card |76795   |1.254208419E7|163.32   |
|boleto      |19784   |2869361.27   |145.03   |
|voucher     |5775    |379436.87    |65.7     |
|debit_card  |1529    |217989.79    |142.57   |
|not_defined |3       |0.0          |0.0      |
+------------+--------+-------------+---------+



**Вывод.** Преобладают платежи кредитной картой: 76 795 записей на сумму около 12,54 млн при среднем платеже 163,32. Число строк таблицы платежей не равно числу заказов, так как один заказ может быть оплачен несколькими платежами.

## 9.1. Позиции заказов, товары и перевод категорий

In [14]:
translations = (
    spark.read
    .option("header", True)
    .csv(str(STAGING / "product_category_name_translation.csv"))
)

sales = (
    items.alias("i")
    .join(
        products_clean.select(
            "product_id", "product_category_name"
        ).alias("p"),
        F.col("i.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(
        translations.alias("t"),
        F.col("p.product_category_name") ==
        F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.col("i.order_id").alias("order_id"),
        F.col("i.order_item_id").alias("order_item_id"),
        F.col("i.product_id").alias("product_id"),
        F.col("i.price").alias("price"),
        F.col("i.freight_value").alias("freight_value"),
        F.coalesce(
            F.col("t.product_category_name_english"),
            F.col("p.product_category_name"),
            F.lit("unknown")
        ).alias("category")
    )
)

print("Строк до JOIN:", items.count())
print("Строк после JOIN:", sales.count())
print(
    "Позиций категории unknown:",
    sales.filter(F.col("category") == "unknown").count()
)

Строк до JOIN: 112650
Строк после JOIN: 112650
Позиций категории unknown: 1603


**Вывод.** Два LEFT JOIN сохранили все 112 650 позиций заказов; число строк до и после соединения совпадает, значит связи «многие ко многим» нет. Для 1 603 позиций категория осталась `unknown`; при INNER JOIN эти позиции были бы потеряны.

## 9.2. Заказы и клиенты: доля поздней доставки по штатам

In [15]:
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "customers.csv"))
)

late_by_state = (
    delivered
    .join(
        customers.select("customer_id", "customer_state"),
        on="customer_id",
        how="left"
    )
    .filter(F.col("order_delivered_customer_date").isNotNull())
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("delivered"),
        F.sum(
            F.when(F.col("is_late"), 1).otherwise(0)
        ).alias("late")
    )
    .withColumn(
        "late_pct",
        F.round(F.col("late") / F.col("delivered") * 100, 2)
    )
    .orderBy(F.desc("late_pct"))
)

late_by_state.show(10, truncate=False)

+--------------+---------+----+--------+
|customer_state|delivered|late|late_pct|
+--------------+---------+----+--------+
|AL            |397      |95  |23.93   |
|MA            |717      |141 |19.67   |
|PI            |476      |76  |15.97   |
|CE            |1279     |196 |15.32   |
|SE            |335      |51  |15.22   |
|BA            |3256     |457 |14.04   |
|RJ            |12350    |1664|13.47   |
|TO            |274      |35  |12.77   |
|PA            |946      |117 |12.37   |
|ES            |1995     |244 |12.23   |
+--------------+---------+----+--------+
only showing top 10 rows


**Вывод.** Наибольшая доля поздней доставки наблюдается в штате AL — 23,93%, однако там всего 397 доставленных заказов. Доли необходимо оценивать вместе с размером группы; результат описывает выборку и не доказывает влияния штата на сроки доставки.

## 10.1. Выручка по товарным категориям через Spark SQL

In [16]:
sales.createOrReplaceTempView("sales")

category_sql = spark.sql('''
SELECT
    category,
    COUNT(*) AS items,
    COUNT(DISTINCT order_id) AS orders,
    ROUND(SUM(price), 2) AS revenue,
    ROUND(AVG(price), 2) AS avg_price
FROM sales
GROUP BY category
ORDER BY revenue DESC
LIMIT 10
''')

category_sql.show(truncate=False)

+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+



**Вывод.** Наибольшая сумма цен позиций приходится на категории `health_beauty` (около 1,259 млн), `watches_gifts` (1,205 млн) и `bed_bath_table` (1,037 млн). Показатель является суммой цен товаров без стоимости доставки, а не полной выручкой по платежам.

## 10.2. Эквивалентная агрегация через DataFrame API

In [17]:
category_sales = (
    sales
    .groupBy("category")
    .agg(
        F.count("*").alias("items"),
        F.countDistinct("order_id").alias("orders"),
        F.round(F.sum("price"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
)

print("Категорий в витрине:", category_sales.count())

category_sales.orderBy(
    F.desc("revenue")
).show(10, truncate=False)

Категорий в витрине: 74
+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+
only showing top 10 rows


**Вывод.** Витрина содержит 74 категории, а первые десять строк совпадают с результатом SQL-запроса. Это подтверждает, что Spark SQL и DataFrame API формируют планы для одного оптимизатора и дают одинаковый результат.

## 11.1. Сохранение витрины в Parquet и CSV

In [18]:
OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

(
    category_sales
    .write
    .mode("overwrite")
    .parquet(str(OUT / "category_sales_parquet"))
)

(
    category_sales
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(OUT / "category_sales_csv"))
)

category_back = spark.read.parquet(
    str(OUT / "category_sales_parquet")
)

print("Строк после round-trip:", category_back.count())
category_back.printSchema()

Строк после round-trip: 74
root
 |-- category: string (nullable = true)
 |-- items: long (nullable = true)
 |-- orders: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- avg_price: double (nullable = true)



**Вывод.** Витрина записана в Parquet и CSV; повторное чтение Parquet вернуло те же 74 строки с сохранёнными числовыми типами, без `inferSchema`. Spark записывает результат в каталог с part-файлами, поскольку разные разделы данных могут записываться параллельно.

## 11.2. Завершение SparkSession

In [19]:
spark.stop()